# Week 2 — Deep Learning 입문 + mini UNet 학습 (1조 Advanced)

## 이번 주 학습 목표
1. **W1 baseline** 의 한계를 명확히 인지 (k가 클수록 |Δφ| 악화)
2. **2D UNet** 아키텍처 핵심 (encoder/decoder + skip-connection) 직접 분석
3. **SliceDataset** 으로 "sparse triplet (before/middle/after)" 학습 데이터 구성
4. **mini UNet** (~30K~120K params) 을 학생 노트북 CPU에서 직접 학습
   - **PRESET 옵션** — `'fast'` (10분), `'standard'` (30분), `'full'` (60분)
5. 학습된 모델의 |Δφ|·SSIM 을 **W1 Linear baseline 과 직접 비교**

## 노트북 사용 방법

본 노트북의 모델과 학습 루프는 helpers/model_utils.py에 정의되어 있습니다. 본문에서는 preset과 파라미터를 바꾸어가며 학습 곡선과 평가 지표 변화를 분석합니다.

본문 **** 블록에서 preset · k · lr · loss 등을 sweep하면서 학습 곡선과 평가 지표의 변화를 관찰. 정답 박스는 없습니다. 본인 노트에 가설·관찰·분석을 자유롭게 기록.

본 노트북은 학생 노트북(CPU)에서 동작합니다. GPU 있으면 자동 사용.

## 0. 환경 준비

In [ ]:
import sys

from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'helpers'))



import numpy as np

import matplotlib.pyplot as plt

import torch



from dr_utils import (

    load_volume, porosity, reconstruct_sparse_linear,

    porosity_error, ssim_3d_mean, summarize_metrics,

    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,

)

from model_utils import (

    UNetMini, count_parameters, SliceDataset,

    train_quick, evaluate_model,

    save_ckpt, load_ckpt, TRAINING_PRESETS,

)

setup_plot_style()



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. W1 baseline 복습 — 우리가 이길 대상



본 W2의 모든 결과는 "W1 Linear baseline 대비 얼마나 좋아졌나" 로 평가합니다.

In [ ]:
DATA = Path('..') / 'data'

bb = load_volume(DATA / 'BB_256.bin')



# W1 B1 (Linear) at k=5

K = 5

rec_l = reconstruct_sparse_linear(bb, k=K)

m_baseline = summarize_metrics(rec_l, bb, label=f'B1 Linear k={K}')

## 2. Mini UNet 아키텍처 분석



UNet 구조:

- **Encoder** (downsampling) — Conv → Pool 반복으로 추상 특징 추출

- **Decoder** (upsampling) — ConvTranspose 로 해상도 복원

- **Skip-connection** — Encoder의 detail을 Decoder에 직접 전달 (UNet의 핵심!)



본 mini UNet:

- 입력: 2-channel `(slice_before, slice_after)` ← W1 linear 보간이 쓰던 "이웃 슬라이스" 2장

- 출력: 1-channel `(slice_middle)` — 가운데 슬라이스 예측 (sigmoid → [0, 1])

- 3 단계 encoder/decoder (depth 3)

In [ ]:
# 세 preset의 모델 크기 비교

print(f"{'preset':<10}{'base':>5}{'params':>10}")

for name, p in TRAINING_PRESETS.items():

    m = UNetMini(in_ch=2, base=p['base'])

    print(f'{name:<10}{p["base"]:>5}{count_parameters(m):>10,}')



# 모델 구조 출력

print('\n--- UNetMini(base=16) 구조 ---')

print(UNetMini(in_ch=2, base=16))

## 3. SliceDataset — sparse triplet 학습 데이터



각 sample: `(input_2ch=[before, after], target_1ch=middle)`



- k=3 → 모든 (before, middle, after) triplet 사용

- patch_size=64 → 256×256 슬라이스에서 64×64 random crop (학습 가속)

- augment=True → flip 증강

In [ ]:
ds = SliceDataset(bb, k=K, patch_size=64, n_patches_per_triplet=4, augment=True)

print(f'Dataset size: {len(ds)} samples')



# 한 sample 시각화

x, y = ds[0]

print(f'x.shape={x.shape}  y.shape={y.shape}')



fig, axes = plt.subplots(1, 3, figsize=(10, 4))

axes[0].imshow(x[0]); axes[0].set_title('Input ch0: before'); axes[0].axis('off')

axes[1].imshow(x[1]); axes[1].set_title('Input ch1: after'); axes[1].axis('off')

axes[2].imshow(y[0]); axes[2].set_title('Target: middle (GT)'); axes[2].axis('off')

plt.tight_layout(); plt.show()

## 4. 학습 — PRESET 선택!



본인 노트북 사양/시간에 맞춰 선택:



| PRESET | base | epochs | 예상 시간 (CPU) | 결과 품질 |

|---|---|---|---|---|

| `'fast'`     | 8  | 20  | ~10 분 | 작동 확인 |

| `'standard'` | 16 | 50  | ~30 분 | 의미있는 비교 |

| `'full'`     | 16 | 100 | ~60 분 | baseline 명확히 능가 |

In [ ]:
PRESET = 'fast'  # ← 본인 환경에 맞춰 변경. 'standard' / 'full' 도 가능.



model, history = train_quick(bb, k=K, preset=PRESET, device=DEVICE, verbose=True)

In [ ]:
# 학습 곡선

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(history, color=ORANGE, lw=2)

ax.set_xlabel('Epoch'); ax.set_ylabel('Train L1 loss')

ax.set_title(f'mini UNet 학습 곡선 (preset={PRESET})')

ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()



# 체크포인트 저장 (나중에 다시 학습 안 해도 됨)

save_ckpt(model, f'unet_mini_{PRESET}.pth',

          meta={'base': TRAINING_PRESETS[PRESET]['base'], 'preset': PRESET, 'k': K})

print(f'✓ saved unet_mini_{PRESET}.pth')

## 5. 평가 — W1 Linear vs UNet mini



전체 BB 부피의 모든 누락 슬라이스를 모델로 복원 → |Δφ|, SSIM 비교.

In [ ]:
res_unet = evaluate_model(model, bb, k=K, device=DEVICE)

print(f'B1 Linear     |Δφ|={m_baseline["dphi"]:.2f}%p   SSIM={m_baseline["ssim"]:.4f}')

print(f'UNet ({PRESET}) |Δφ|={res_unet["dphi_pp"]:.2f}%p   SSIM={res_unet["ssim"]:.4f}')



improvement = (m_baseline['dphi'] - res_unet['dphi_pp']) / m_baseline['dphi'] * 100

print(f'\n|Δφ| 개선: {improvement:+.1f}% (양수면 UNet이 더 좋음)')

In [ ]:
# 시각: z=62에서 GT vs B1 vs UNet

z_show = 62

recon_unet = res_unet['recon']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(bb[z_show]); axes[0].set_title(f'GT z={z_show}')

axes[1].imshow(rec_l[z_show]); axes[1].set_title('B1 Linear')

axes[2].imshow(recon_unet[z_show]); axes[2].set_title(f'UNet ({PRESET})')

diff = np.abs(bb[z_show].astype(float) - recon_unet[z_show])

axes[3].imshow(diff, cmap='hot'); axes[3].set_title('|GT − UNet|')

for ax in axes: ax.axis('off')

plt.tight_layout(); plt.show()

## 6. 박스 모음



### PRESET 변경 → 학습 시간 vs 정확도

위 4번 셀에서 `PRESET = 'standard'` 또는 `'full'` 로 바꿔 재학습.

**비교 표 작성:** preset별 시간 / 파라미터 / |Δφ| / SSIM 4-row.



### sparse k 변경

본 노트북은 K=5 학습. K=3 또는 K=7로 바꿔 학습/평가.

- K=3 (쉬움) → UNet의 |Δφ| 가 baseline 대비 얼마나 작아지나?

- K=7 (어려움) → UNet도 baseline 정도로 떨어지지 않나?



### 다른 도메인에서 평가

BB에서 학습한 모델을 CastleGate, Bentheimer, Parker 에 평가 (zero-shot 비슷).

도메인 generalization 어느 정도?

In [ ]:
# [Try-it! ③] 예시 — BB 학습 모델을 다른 도메인에 평가

for name in ['CastleGate', 'Bentheimer', 'Parker']:

    vol = load_volume(DATA / f'{name}_256.bin')

    res = evaluate_model(model, vol, k=K, device=DEVICE)

    print(f'  {name:12s}  UNet |Δφ|={res["dphi_pp"]:.2f}%p  SSIM={res["ssim"]:.4f}')

## 7. 다음 주 (W3) — 손실 함수 + HPO 입문

- 본 W2는 L1 loss만 사용. SSIM / porosity / surface-area 손실 추가 시 결과 변화 탐구
- Loss weight를 어떻게 고르나? → Optuna multi-objective HPO 결과 분석
- 1조: mini Optuna (3 trial, ~20분) 직접 실행
- `pip install pytorch-msssim optuna`

---

## 🎯 W2 탐구 과제 (1조 Advanced)

본 노트북을 본인 작업 파일로 복사한 뒤, 다음 과제를 본인 분석·시각화·해석과 함께 정리해 제출.

### 과제 1 — preset 비교 (필수)

`'fast'` 와 `'standard'` 두 preset으로 학습 → 시간 / 파라미터 / |Δφ| / SSIM 비교. 학습 곡선까지 함께 비교 + 본인 해석 (왜 더 큰 모델이 항상 좋지 않을 수 있는가?).

### 과제 2 — Cross-domain 일반화 (필수)

학습된 모델을 4 도메인에 평가. cross-domain generalization 결과를 표 + 시각화로 정리. 차이의 원인에 대한 본인 가설 (도메인별 공극률·구조·등방성 차이 등) 제시.

### 과제 3 — Loss 종류 비교 (선택, 도전)

`criterion = nn.L1Loss()` 를 `nn.MSELoss()` 또는 다른 손실로 바꿔 같은 preset으로 재학습. 결과 차이를 본인 분석. L1 / L2 가 각각 어떤 상황에서 유리할지 본인 해석.

### 과제 4 — Learning rate sensitivity (선택, 심화)

`train_quick` 내부 `Adam(lr=1e-3)` 의 lr ∈ {1e-4, 5e-4, 1e-3, 5e-3} sweep + 학습 곡선 비교 + 본인 해석. 학습률이 너무 작거나 너무 크면 학습이 어떻게 망가지는지 본인 관찰을 정리.